# Projeto Final — Análise Exploratória de Dados (Olist)

## 1. Apresentação da Base e Perguntas de Negócio
### Origem dos Dados
Os dados foram extraídos do dataset público do **Olist** no Kaggle, cobrindo o e-commerce brasileiro.

### Perguntas de Negócio
1. **Prazos e Frete:** Qual a relação entre o custo do frete e o tempo real de entrega entre os diferentes estados?
2. **Meios de Pagamento e Ticket:** Como o valor médio do pedido varia em função do tipo de pagamento e do parcelamento?
3. **Outliers de Frete:** Quais estados apresentam os maiores outliers de valor de frete em relação ao preço das mercadorias?
4. **Recorrência:** Clientes recorrentes possuem comportamento de compra diferente dos clientes pontuais?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Leitura das bases de dados
df_customers = pd.read_csv('../dados/olist_customers_dataset.csv')
df_orders = pd.read_csv('../dados/olist_orders_dataset.csv')
df_items = pd.read_csv('../dados/olist_order_items_dataset.csv')
df_payments = pd.read_csv('../dados/olist_order_payments_dataset.csv')

print("Bases carregadas com sucesso!")

Realizamos a junção das 4 bases para criar o DataFrame consolidado da análise.

In [ ]:
# Unificação das tabelas
df = df_orders.merge(df_customers, on='customer_id', how='inner') \
              .merge(df_items, on='order_id', how='inner') \
              .merge(df_payments, on='order_id', how='inner')

print(f"Shape do Dataset Consolidado: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head(3)

## 2. Diagnóstico de Qualidade dos Dados
Dimensões, tipos, uso de memória

#### 2.1 Dimensões e Tipos

In [ ]:
df.info()

Faltantes por coluna (quantidade e percentual)

In [ ]:
# Verificação de Faltantes
nulos = pd.DataFrame({
    'Total_Nulos': df.isnull().sum(),
    'Percentual_%': (df.isnull().mean() * 100).round(2)
})
print(nulos[nulos['Total_Nulos'] > 0].sort_values(by='Total_Nulos', ascending=False))

Duplicados, categorias inconsistentes, valores inválidos

In [ ]:
# Verificação de Duplicados
print(f"\nLinhas duplicadas no dataset consolidado: {df.duplicated().sum()}")

### 2.2 Identificação de Outliers

#### Método IQR

In [ ]:
# Seleciona apenas colunas numéricas
colunas_numericas = df.select_dtypes(include="number").columns

# Opcional: remova identificadores ou códigos que não devem ser analisados
# colunas_numericas = colunas_numericas.drop(["order_item_id"])

outliers = pd.DataFrame(False, index=df.index, columns=colunas_numericas)
limites = {}

for coluna in colunas_numericas:
    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    limites[coluna] = (limite_inferior, limite_superior)

    outliers[coluna] = (
        (df[coluna] < limite_inferior) |
        (df[coluna] > limite_superior)
    )

# Quantidade de outliers por coluna
quantidade_outliers = outliers.sum().sort_values(ascending=False)
print(quantidade_outliers[quantidade_outliers > 0])

In [ ]:
df_outliers = df[outliers.any(axis=1)]

print(df_outliers.shape)
display(df_outliers.head())

In [ ]:
df[colunas_numericas].boxplot(figsize=(14, 6), rot=45)
plt.title("Boxplots das variáveis numéricas")
plt.show()

#### Método Z-Score

In [ ]:
# Seleciona apenas colunas numéricas
colunas_numericas = df.select_dtypes(include="number").columns

# Opcional: remova identificadores ou códigos que não devem ser analisados
# colunas_numericas = colunas_numericas.drop(["order_item_id"])

# Calcula o z-score de cada valor em relação à sua coluna
media = df[colunas_numericas].mean()
desvio_padrao = df[colunas_numericas].std()
z_scores = (df[colunas_numericas] - media) / desvio_padrao

# Considera outlier todo valor com z-score absoluto maior que 3
limite_z = 3
outliers_z = z_scores.abs() > limite_z

# Quantidade de outliers por coluna
quantidade_outliers_z = outliers_z.sum().sort_values(ascending=False)
print(quantidade_outliers_z[quantidade_outliers_z > 0])

# Linhas que possuem pelo menos um outlier
df_outliers_z = df[outliers_z.any(axis=1)]
print(f"\\nLinhas com pelo menos um outlier: {df_outliers_z.shape[0]}")
display(df_outliers_z.head())

Próximo passo: Analisar a distribuição de cada variável quantitativa para determinar o tipo (normal ou assimétrica)

## 3 Limpeza e transformação

## 4 Análise exploratória

## 5 Conclusões